# NOTEBOOK 00.2: *METEOROLOGÍA*

In [1]:
import pandas as pd
import requests
import time

# Webscraping de datos meteorológicos

Decidimos extraer los datos del NOAA (National Oceanic and Atmospheric Administration), un centro oficial del Gobierno de Estados Unidos.

Tras explorar la web conseguimos los códigos de las estaciones meteorológicas adecuadas y decidimos la siguiente lista de variables a descargar:
- TMAX: Temperatura máxima
- TMIN: Temperatura mínima
- PRCP: Cantidad de lluvia caída
- SNOW: Cantidad de nieve caída
- SNWD: Espesor de nieve
- AWND: Velocidad del viento

In [2]:
API_KEY = "RjuUZNmFgnTIOqrzpGeqZJBoUPdamgiW"

In [3]:
url = "https://www.ncdc.noaa.gov/cdo-web/api/v2/data"

cities = {
    "New York": "GHCND:USW00094728",
    "Boston": "GHCND:USW00014739",
    "Philadelphia": "GHCND:USW00013739"
}

variables = ['AWND', 'PRCP', 'SNOW', 'SNWD', 'TMAX', 'TMIN']

years = range(2011, 2017)

# Segmentamos la query en trimestres para no superar el límite de descarga
trimestres = [
    ("01-01", "03-31"),
    ("04-01", "06-30"),
    ("07-01", "09-30"),
    ("10-01", "12-31")
]

data_list = []

for city, station in cities.items():
    for year in years:
        for trimestre in trimestres:
            start_date = f"{year}-{trimestre[0]}"
            end_date = f"{year}-{trimestre[1]}"
            print(f"📡 Obteniendo datos para {city} ({start_date} a {end_date})...")

            params = {
                "datasetid": "GHCND",
                "stationid": station,
                "startdate": start_date,
                "enddate": end_date,
                "limit": 1000,
                "units": "metric",
                "datatypeid": variables
            }

            headers = {"token": API_KEY}

            intento = 0
            max_intentos = 5
            exito = False

            while intento < max_intentos and not exito:
                response = requests.get(url, headers=headers, params=params)

                if response.status_code == 200:
                    data = response.json()
                    if "results" in data:
                        for entry in data["results"]:
                            data_list.append({
                                "region": city,
                                "date": entry["date"][:10],
                                "datatype": entry["datatype"],
                                "value": entry["value"]
                            })
                    else:
                        print(f"⚠️ No hay datos disponibles para {city} ({start_date} a {end_date}).")
                    print(f"✅ Obtenidos datos para {city} ({start_date} a {end_date})")
                    exito = True
                else:
                    intento += 1
                    print(f"❌ Error {response.status_code} en {city} ({start_date} a {end_date}), intento {intento}/{max_intentos}")
                    print("🔍 Respuesta de la API:", response.text)
                    
                    if intento < max_intentos:
                        time.sleep(2 ** intento)  # Espera exponencial antes de reintentar
                    else:
                        print(f"⛔ Fallo definitivo en {city} ({start_date} a {end_date}), se omitirá.")

df_weather = pd.DataFrame(data_list)    # Guardamos en un dataframe toda la información

📡 Obteniendo datos para New York (2011-01-01 a 2011-03-31)...
✅ Obtenidos datos para New York (2011-01-01 a 2011-03-31)
📡 Obteniendo datos para New York (2011-04-01 a 2011-06-30)...
✅ Obtenidos datos para New York (2011-04-01 a 2011-06-30)
📡 Obteniendo datos para New York (2011-07-01 a 2011-09-30)...
✅ Obtenidos datos para New York (2011-07-01 a 2011-09-30)
📡 Obteniendo datos para New York (2011-10-01 a 2011-12-31)...
✅ Obtenidos datos para New York (2011-10-01 a 2011-12-31)
📡 Obteniendo datos para New York (2012-01-01 a 2012-03-31)...
✅ Obtenidos datos para New York (2012-01-01 a 2012-03-31)
📡 Obteniendo datos para New York (2012-04-01 a 2012-06-30)...
✅ Obtenidos datos para New York (2012-04-01 a 2012-06-30)
📡 Obteniendo datos para New York (2012-07-01 a 2012-09-30)...
✅ Obtenidos datos para New York (2012-07-01 a 2012-09-30)
📡 Obteniendo datos para New York (2012-10-01 a 2012-12-31)...
✅ Obtenidos datos para New York (2012-10-01 a 2012-12-31)
📡 Obteniendo datos para New York (2013-0

In [4]:
df_weather.head()

,region,date,datatype,value
0,New York,2011-01-01,AWND,1.4
1,New York,2011-01-01,PRCP,0.0
2,New York,2011-01-01,SNOW,0.0
3,New York,2011-01-01,SNWD,305.0
4,New York,2011-01-01,TMAX,11.7


In [5]:
# Comprobamos que la temperatura está en Celsius
# La temperatura más alta registrada es el 22 de julio de 2011 en NYC, y son 40ºC (40ºF equivaldrían a 4ºC, así que los datos no pueden estar en esa escala)
df_weather[(df_weather['datatype'] == 'TMAX') & (df_weather['value'] >= 40)]

,region,date,datatype,value
1214,New York,2011-07-22,TMAX,40.0


In [6]:
df_weather['date'] = pd.to_datetime(df_weather['date'])

In [7]:
df_weather.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 37244 entries, 0 to 37243
Data columns (total 4 columns):
 #   Column    Non-Null Count  Dtype         
---  ------    --------------  -----         
 0   region    37244 non-null  object        
 1   date      37244 non-null  datetime64[ns]
 2   datatype  37244 non-null  object        
 3   value     37244 non-null  float64       
dtypes: datetime64[ns](1), float64(1), object(2)
memory usage: 1.1+ MB


In [8]:
df_weather.isnull().sum()

region      0
date        0
datatype    0
value       0
dtype: int64

In [9]:
df_pivot = df_weather.pivot(index=["region", "date"], columns="datatype", values="value").reset_index()


In [10]:
df_pivot

datatype,region,date,AWND,PRCP,SNOW,SNWD,TMAX,TMIN
0,Boston,2011-01-01,2.7,0.0,0.0,NaN,13.3,1.7
1,Boston,2011-01-02,3.1,3.8,0.0,NaN,10.6,3.3
2,Boston,2011-01-03,6.9,0.0,0.0,NaN,3.3,-1.7
3,Boston,2011-01-04,4.6,0.0,0.0,NaN,3.9,-3.9
4,Boston,2011-01-05,5.2,0.0,0.0,NaN,3.9,-2.2
...,...,...,...,...,...,...,...,...
6571,Philadelphia,2016-12-27,6.4,0.8,0.0,0.0,17.8,7.2
6572,Philadelphia,2016-12-28,3.8,0.0,0.0,0.0,7.8,-0.5
6573,Philadelphia,2016-12-29,4.4,4.8,0.0,0.0,7.8,-1.0
6574,Philadelphia,2016-12-30,7.3,0.0,0.0,0.0,5.6,0.0


In [11]:
df_pivot.columns.name = None    # Quitamos el nombre a la columna de index
df_pivot.rename(columns={
    "AWND": "wind_speed",
    "PRCP": "precipitation",
    "SNOW": "snowfall",
    "SNWD": "snow_depth",
    "TMAX": "temp_max",
    "TMIN": "temp_min"
}, inplace=True)

In [12]:
df_pivot

,region,date,wind_speed,precipitation,snowfall,snow_depth,temp_max,temp_min
0,Boston,2011-01-01,2.7,0.0,0.0,NaN,13.3,1.7
1,Boston,2011-01-02,3.1,3.8,0.0,NaN,10.6,3.3
2,Boston,2011-01-03,6.9,0.0,0.0,NaN,3.3,-1.7
3,Boston,2011-01-04,4.6,0.0,0.0,NaN,3.9,-3.9
4,Boston,2011-01-05,5.2,0.0,0.0,NaN,3.9,-2.2
...,...,...,...,...,...,...,...,...
6571,Philadelphia,2016-12-27,6.4,0.8,0.0,0.0,17.8,7.2
6572,Philadelphia,2016-12-28,3.8,0.0,0.0,0.0,7.8,-0.5
6573,Philadelphia,2016-12-29,4.4,4.8,0.0,0.0,7.8,-1.0
6574,Philadelphia,2016-12-30,7.3,0.0,0.0,0.0,5.6,0.0


In [13]:
df_pivot[(df_pivot['snow_depth'].isnull() == True) & (df_pivot['region'] != 'Boston')]

,region,date,wind_speed,precipitation,snowfall,snow_depth,temp_max,temp_min


In [14]:
#Como no hay registros del espesor de la nieve en Boston, eliminamos la variable
df_pivot.drop(labels='snow_depth', axis=1, inplace=True)

In [15]:
df_pivot

,region,date,wind_speed,precipitation,snowfall,temp_max,temp_min
0,Boston,2011-01-01,2.7,0.0,0.0,13.3,1.7
1,Boston,2011-01-02,3.1,3.8,0.0,10.6,3.3
2,Boston,2011-01-03,6.9,0.0,0.0,3.3,-1.7
3,Boston,2011-01-04,4.6,0.0,0.0,3.9,-3.9
4,Boston,2011-01-05,5.2,0.0,0.0,3.9,-2.2
...,...,...,...,...,...,...,...
6571,Philadelphia,2016-12-27,6.4,0.8,0.0,17.8,7.2
6572,Philadelphia,2016-12-28,3.8,0.0,0.0,7.8,-0.5
6573,Philadelphia,2016-12-29,4.4,4.8,0.0,7.8,-1.0
6574,Philadelphia,2016-12-30,7.3,0.0,0.0,5.6,0.0


In [16]:
# Creamos la variable week y yearweek para posterior merge con el df principal
df_pivot['week'] = df_pivot['date'].dt.to_period('W').apply(lambda r: r.start_time)
df_pivot['week'] = pd.to_datetime(df_pivot['week'])

df_pivot['yearweek'] = df_pivot['week'].dt.isocalendar().year.astype(str) + df_pivot['week'].dt.isocalendar().week.astype(str).str.zfill(2)

In [17]:
# Agrupamos por semana. Para cada variable calculamos su valor máximo, mínimo y medio
df_weekly = df_pivot.groupby(['region','yearweek']).agg({
    "wind_speed": ["max", "min", "mean"],
    "precipitation": ["max", "min", "mean"],
    "snowfall": ["max", "min", "mean"],
    "temp_max": ["max", "min", "mean"],
    "temp_min": ["max", "min", "mean"]
}).reset_index()

In [18]:
df_weekly.head()

region yearweek wind_speed                precipitation                 \
                          max  min      mean           max  min      mean   
0  Boston   201052        3.1  2.7  2.900000           3.8  0.0  1.900000   
1  Boston   201101        7.5  2.8  5.171429           4.1  0.0  1.128571   
2  Boston   201102       10.6  2.9  6.114286          35.6  0.0  5.085714   
3  Boston   201103        6.3  3.6  4.557143          24.1  0.0  6.771429   
4  Boston   201104        6.5  2.3  3.871429          17.3  0.0  3.057143   

  snowfall                 temp_max                  temp_min                  
       max  min       mean      max   min       mean      max   min      mean  
0      0.0  0.0   0.000000     13.3  10.6  11.950000      3.3   1.7  2.500000  
1     48.0  0.0  13.857143      3.9   0.0   2.157143     -1.7  -5.6 -3.585714  
2    371.0  0.0  53.000000      1.7  -3.9   0.085714     -2.2  -9.4 -6.100000  
3    185.0  0.0  33.285714      4.4  -6.1  -0.942857      0.0 -15.0 -7.142857  
4    224.0  0.0  38.857143      2.8 -10.6  -0.314286     -3.3 -18.9 -8.314286

In [19]:
# Acomodamos los nombres de las columnas
df_weekly.columns = [
    "region",
    "yearweek",
    "max_wind",
    "min_wind", 
    "avg_wind", 
    "max_rain", 
    "min_rain", 
    "avg_rain", 
    "max_snow", 
    "min_snow", 
    "avg_snow", 
    "max_tempmax", 
    "min_tempmax", 
    "avg_tempmax", 
    "max_tempmin", 
    "min_tempmin", 
    "avg_tempmin"
]


In [20]:
df_weekly.head()

,region,yearweek,max_wind,min_wind,avg_wind,max_rain,min_rain,avg_rain,max_snow,min_snow,avg_snow,max_tempmax,min_tempmax,avg_tempmax,max_tempmin,min_tempmin,avg_tempmin
0,Boston,201052,3.1,2.7,2.900000,3.8,0.0,1.900000,0.0,0.0,0.000000,13.3,10.6,11.950000,3.3,1.7,2.500000
1,Boston,201101,7.5,2.8,5.171429,4.1,0.0,1.128571,48.0,0.0,13.857143,3.9,0.0,2.157143,-1.7,-5.6,-3.585714
2,Boston,201102,10.6,2.9,6.114286,35.6,0.0,5.085714,371.0,0.0,53.000000,1.7,-3.9,0.085714,-2.2,-9.4,-6.100000
3,Boston,201103,6.3,3.6,4.557143,24.1,0.0,6.771429,185.0,0.0,33.285714,4.4,-6.1,-0.942857,0.0,-15.0,-7.142857
4,Boston,201104,6.5,2.3,3.871429,17.3,0.0,3.057143,224.0,0.0,38.857143,2.8,-10.6,-0.314286,-3.3,-18.9,-8.314286


In [21]:
df_weekly.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 942 entries, 0 to 941
Data columns (total 17 columns):
 #   Column       Non-Null Count  Dtype  
---  ------       --------------  -----  
 0   region       942 non-null    object 
 1   yearweek     942 non-null    object 
 2   max_wind     942 non-null    float64
 3   min_wind     942 non-null    float64
 4   avg_wind     942 non-null    float64
 5   max_rain     942 non-null    float64
 6   min_rain     942 non-null    float64
 7   avg_rain     942 non-null    float64
 8   max_snow     942 non-null    float64
 9   min_snow     942 non-null    float64
 10  avg_snow     942 non-null    float64
 11  max_tempmax  942 non-null    float64
 12  min_tempmax  942 non-null    float64
 13  avg_tempmax  942 non-null    float64
 14  max_tempmin  942 non-null    float64
 15  min_tempmin  942 non-null    float64
 16  avg_tempmin  942 non-null    float64
dtypes: float64(15), object(2)
memory usage: 125.2+ KB


In [22]:
# Comprobamos que no hay fechas duplicadas para ninguna ciudad
df_weekly[['yearweek', 'region']].duplicated().sum()

np.int64(0)

In [23]:
df_weekly.to_csv('data/meteo_semanal.csv')